In [ ]:
# CELL 1 — Imports & Style Setup
# Loads all libraries and applies a dark survey/research chart theme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F0F1A', 'axes.facecolor':   '#161625',
    'axes.edgecolor':   '#252540', 'axes.labelcolor':  '#C8C8E8',
    'xtick.color':      '#7070A0', 'ytick.color':      '#7070A0',
    'grid.color':       '#252540', 'grid.alpha':        0.6,
    'text.color':       '#E8E8FF', 'font.family':      'DejaVu Sans',
    'axes.titlepad':    14,        'axes.titlesize':    12,
    'axes.titleweight': 'bold',
})
Q_COLORS   = ['#7C83FD','#96EFFF','#FFD166','#EF476F','#06D6A0','#FF9A3C']
REG_COLORS = {'North':'#7C83FD','South':'#06D6A0','East':'#FFD166','West':'#EF476F','Central':'#96EFFF'}
AGE_COLORS = {'18-24':'#7C83FD','25-34':'#96EFFF','35-44':'#FFD166','45-54':'#EF476F','55+':'#06D6A0'}
GEN_COLORS = {'Male':'#7C83FD','Female':'#EF476F','Non-binary':'#FFD166','Prefer not to say':'#96EFFF'}
print("✅ Libraries loaded. Survey theme configured.")

In [ ]:
# CELL 2 — Build Synthetic Poll Dataset
# Simulates 1,200 survey responses across 5 questions, 5 regions, 5 age groups.
# Realistic probability weights — Python leads Q1, Hybrid leads Q2, AI/ML leads Q3.
np.random.seed(42)
N = 1200

regions    = ['North','South','East','West','Central']
age_groups = ['18-24','25-34','35-44','45-54','55+']
genders    = ['Male','Female','Non-binary','Prefer not to say']
edu_levels = ['High School','Bachelor','Master','PhD']
devices    = ['Mobile','Desktop','Tablet']

questions = {
    'Q1: Best programming language for beginners?':
        {'Python':0.48,'JavaScript':0.22,'Java':0.14,'C++':0.09,'Others':0.07},
    'Q2: Preferred work mode?':
        {'Remote':0.38,'Hybrid':0.41,'On-site':0.21},
    'Q3: Most important skill in 2024?':
        {'AI/ML':0.35,'Cloud Computing':0.25,'Cybersecurity':0.20,'Data Analytics':0.15,'Blockchain':0.05},
    'Q4: Preferred learning platform?':
        {'YouTube':0.30,'Coursera':0.24,'Udemy':0.22,'LinkedIn Learning':0.14,'Others':0.10},
    'Q5: Job satisfaction level?':
        {'Very Satisfied':0.22,'Satisfied':0.35,'Neutral':0.25,'Dissatisfied':0.13,'Very Dissatisfied':0.05},
}

dates = pd.date_range('2024-01-01', periods=N, freq='6h')
rows  = []
for i in range(N):
    region = np.random.choice(regions,    p=[0.22,0.20,0.18,0.22,0.18])
    age    = np.random.choice(age_groups, p=[0.20,0.28,0.22,0.18,0.12])
    gender = np.random.choice(genders,    p=[0.48,0.44,0.04,0.04])
    edu    = np.random.choice(edu_levels, p=[0.20,0.40,0.28,0.12])
    device = np.random.choice(devices,    p=[0.55,0.35,0.10])
    row = {'respondent_id':f'R{i+1:04d}','date':dates[i].date(),
           'region':region,'age_group':age,'gender':gender,'education':edu,'device':device}
    for q, opts in questions.items():
        row[q] = np.random.choice(list(opts.keys()), p=list(opts.values()))
    rows.append(row)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df.to_csv('poll_data.csv', index=False)
q_cols = list(questions.keys())
total  = len(df)

print(f"✅ Poll dataset: {total:,} respondents × {df.shape[1]} columns")
print(f"   Questions : {len(questions)}")
print(f"   Regions   : {df['region'].nunique()}  |  Age Groups: {df['age_group'].nunique()}")
print(f"   Period    : {df['date'].min().date()} → {df['date'].max().date()}")
df.head(6)

In [ ]:
# CELL 3 — Data Overview & Response Summary
print("="*60)
print("  POLL DATA OVERVIEW")
print("="*60)
print(f"\n🔢 Shape       : {df.shape}")
print(f"❌ Null values  : {df.isnull().sum().sum()}")
print(f"\n📊 Respondent Profile:")
print(f"  Regions  : {dict(df['region'].value_counts())}")
print(f"  Devices  : {dict(df['device'].value_counts())}")
print(f"\n🗳️  Vote Counts per Question:")
for q in q_cols:
    winner = df[q].value_counts().idxmax()
    pct    = df[q].value_counts().max()/total*100
    print(f"  {q[:45]}")
    print(f"    → Winner: '{winner}' ({pct:.1f}%)")

In [ ]:
# CELL 4 — Main Poll Dashboard (All 5 Questions)
# One combined figure showing all 5 poll questions with different chart types.
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0F0F1A')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.40)

# Q1 — horizontal bar (flagship question)
ax0 = fig.add_subplot(gs[0, :2]); ax0.set_facecolor('#161625')
q1_counts = df[q_cols[0]].value_counts().sort_values()
bars = ax0.barh(q1_counts.index, q1_counts.values,
                color=Q_COLORS[:len(q1_counts)], alpha=0.88, edgecolor='none', height=0.55)
ax0.set_title(f'🗳️  {q_cols[0]}', color='#E8E8FF', fontsize=12)
ax0.set_xlabel('Votes')
ax0.grid(True, axis='x', alpha=0.25)
ax0.spines['top'].set_visible(False); ax0.spines['right'].set_visible(False)
for bar, val in zip(bars, q1_counts.values):
    ax0.text(val+5, bar.get_y()+bar.get_height()/2,
             f'{val} ({val/total*100:.1f}%)', va='center', color='#C8C8E8', fontsize=10)

# Q2 — donut
ax1 = fig.add_subplot(gs[0, 2]); ax1.set_facecolor('#161625')
q2_counts = df[q_cols[1]].value_counts()
wedges, texts, autotexts = ax1.pie(
    q2_counts.values, labels=q2_counts.index, autopct='%1.1f%%',
    colors=Q_COLORS[:len(q2_counts)], startangle=90,
    wedgeprops=dict(width=0.58, edgecolor='#0F0F1A', linewidth=1.8),
    textprops={'color':'#C8C8E8','fontsize':9}, pctdistance=0.78)
for at in autotexts: at.set_fontsize(8); at.set_color('#0F0F1A'); at.set_fontweight('bold')
ax1.set_title(f'🥧  {q_cols[1]}', color='#E8E8FF', fontsize=11)

# Q3 — vertical bar
ax2 = fig.add_subplot(gs[1, 0]); ax2.set_facecolor('#161625')
q3_counts = df[q_cols[2]].value_counts().sort_values(ascending=False)
bars3 = ax2.bar(q3_counts.index, q3_counts.values,
                color=Q_COLORS[:len(q3_counts)], alpha=0.88, edgecolor='none', width=0.6)
for b, v in zip(bars3, q3_counts.values):
    ax2.text(b.get_x()+b.get_width()/2, v+5, f'{v/total*100:.0f}%',
             ha='center', color='#C8C8E8', fontsize=9)
ax2.set_title(f'🔑  {q_cols[2]}', color='#E8E8FF', fontsize=10)
ax2.set_ylabel('Votes'); ax2.tick_params(axis='x', rotation=18)
ax2.grid(True, axis='y', alpha=0.25)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# Q4 — horizontal bar
ax3 = fig.add_subplot(gs[1, 1]); ax3.set_facecolor('#161625')
q4_counts = df[q_cols[3]].value_counts().sort_values()
ax3.barh(q4_counts.index, q4_counts.values,
         color=Q_COLORS[:len(q4_counts)], alpha=0.88, edgecolor='none', height=0.55)
ax3.set_title(f'📚  {q_cols[3]}', color='#E8E8FF', fontsize=10)
ax3.set_xlabel('Votes')
ax3.grid(True, axis='x', alpha=0.25)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
for bar, val in zip(ax3.patches, q4_counts.values):
    ax3.text(val+3, bar.get_y()+bar.get_height()/2,
             f'{val/total*100:.1f}%', va='center', color='#C8C8E8', fontsize=9)

# Q5 — satisfaction diverging bar
ax4 = fig.add_subplot(gs[1, 2]); ax4.set_facecolor('#161625')
q5_order  = ['Very Satisfied','Satisfied','Neutral','Dissatisfied','Very Dissatisfied']
q5_colors = ['#06D6A0','#96EFFF','#FFD166','#FF9A3C','#EF476F']
q5_counts = df[q_cols[4]].value_counts().reindex(q5_order)
bars5 = ax4.bar(q5_order, q5_counts.values, color=q5_colors, alpha=0.88, edgecolor='none', width=0.6)
for b, v in zip(bars5, q5_counts.values): ax4.text(b.get_x()+b.get_width()/2, v+4, f'{v}', ha='center', color='#C8C8E8', fontsize=9)
ax4.set_title('😊  Q5: Job Satisfaction', color='#E8E8FF', fontsize=11)
ax4.set_ylabel('Votes'); ax4.tick_params(axis='x', rotation=20)
ax4.grid(True, axis='y', alpha=0.25)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

fig.suptitle(f'📊  Poll Results Visualizer — Complete Survey Dashboard (N={total:,})',
             fontsize=15, color='#E8E8FF', y=1.01)
plt.tight_layout()
plt.savefig('chart1_poll_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0F0F1A')
plt.show()
print("💾 Saved: chart1_poll_dashboard.png")

In [ ]:
# CELL 5 — Regional Analysis (Stacked Bar + Respondent Count)
# Shows how Q1 answers vary by region — reveals geographic opinion differences.
q1_options = list(questions[q_cols[0]].keys())
region_q1  = pd.crosstab(df['region'], df[q_cols[0]], normalize='index') * 100
region_q1  = region_q1.reindex(columns=q1_options)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Stacked percentage bar
bottom = np.zeros(len(region_q1))
for j, opt in enumerate(q1_options):
    vals = region_q1[opt].values
    bars = ax1.bar(region_q1.index, vals, bottom=bottom,
                   color=Q_COLORS[j], alpha=0.88, edgecolor='#0F0F1A', linewidth=0.5,
                   label=opt, width=0.6)
    for bar, v, b in zip(bars, vals, bottom):
        if v > 6:
            ax1.text(bar.get_x()+bar.get_width()/2, b+v/2,
                     f'{v:.0f}%', ha='center', va='center',
                     color='#0F0F1A', fontsize=8, fontweight='bold')
    bottom += vals
ax1.set_title('🌍  Q1 Responses by Region (% Stacked)', fontsize=12)
ax1.set_ylabel('Percentage (%)'); ax1.set_xlabel('Region')
ax1.legend(loc='upper right', framealpha=0.15, labelcolor='white', fontsize=9)
ax1.set_ylim(0, 105)
ax1.grid(True, axis='y', alpha=0.2)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Region respondent count
reg_counts = df['region'].value_counts().sort_values(ascending=True)
ax2.barh(reg_counts.index, reg_counts.values,
         color=[REG_COLORS[r] for r in reg_counts.index], alpha=0.88, edgecolor='none')
for bar, val in zip(ax2.patches, reg_counts.values):
    ax2.text(val+5, bar.get_y()+bar.get_height()/2,
             f'{val} ({val/total*100:.1f}%)', va='center', color='#C8C8E8', fontsize=10)
ax2.set_title('👥  Respondent Count by Region', fontsize=12)
ax2.set_xlabel('Number of Respondents')
ax2.grid(True, axis='x', alpha=0.25)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart2_regional_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0F0F1A')
plt.show()
print("💾 Saved: chart2_regional_analysis.png")

In [ ]:
# CELL 6 — Demographic Heatmaps (Age × Q1 and Gender × Q2)
# Shows how different demographics answer differently — key insight for research roles.
q1_options = list(questions[q_cols[0]].keys())
age_q1 = pd.crosstab(df['age_group'], df[q_cols[0]], normalize='index') * 100
age_q1 = age_q1[q1_options]
gen_q2 = pd.crosstab(df['gender'], df[q_cols[1]], normalize='index') * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(age_q1.round(1), annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.4, linecolor='#0F0F1A', ax=ax1,
            cbar_kws={'label':'% of Age Group'})
ax1.set_title('🎂  Q1 Response % by Age Group', fontsize=12, pad=12)
ax1.set_xlabel('Option'); ax1.set_ylabel('Age Group')
ax1.tick_params(axis='x', rotation=20)

sns.heatmap(gen_q2.round(1), annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.4, linecolor='#0F0F1A', ax=ax2,
            cbar_kws={'label':'% of Gender Group'})
ax2.set_title('⚧  Q2 Response % by Gender', fontsize=12, pad=12)
ax2.set_xlabel('Option'); ax2.set_ylabel('Gender')
ax2.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('chart3_demographic_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0F0F1A')
plt.show()
print("💾 Saved: chart3_demographic_heatmap.png")

In [ ]:
# CELL 7 — Feature Engineering
# Adds useful derived columns for deeper analysis.
df['is_mobile']    = (df['device'] == 'Mobile').astype(int)
df['is_satisfied'] = df[q_cols[4]].isin(['Very Satisfied','Satisfied']).astype(int)
df['age_numeric']  = df['age_group'].map({'18-24':21,'25-34':29,'35-44':39,'45-54':49,'55+':60})

print("✅ New features added:")
print(f"  is_mobile    → {df['is_mobile'].sum()} mobile respondents ({df['is_mobile'].mean()*100:.1f}%)")
print(f"  is_satisfied → {df['is_satisfied'].sum()} satisfied respondents ({df['is_satisfied'].mean()*100:.1f}%)")
print(f"\n📊 Cross-tab: Work Mode preference by Age Group:")
print(pd.crosstab(df['age_group'], df[q_cols[1]], normalize='index').round(3) * 100)

In [ ]:
# CELL 8 — Response Trend Over Time (Weekly Volume + Cumulative)
# Shows survey engagement over the collection period.
weekly_counts = df.groupby('week').size().reset_index(name='responses')
cumulative    = weekly_counts['responses'].cumsum()
avg_r         = weekly_counts['responses'].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(weekly_counts['week'], weekly_counts['responses'],
         color='#7C83FD', linewidth=2.2, marker='o', markersize=5,
         markerfacecolor='#FFD166', markeredgecolor='white', markeredgewidth=1)
ax1.fill_between(weekly_counts['week'], weekly_counts['responses'], alpha=0.12, color='#7C83FD')
ax1.axhline(y=avg_r, color='#FFD166', linestyle='--', linewidth=1.3, alpha=0.7,
            label=f'Avg: {avg_r:.0f}/week')
ax1.set_title('📈  Weekly Survey Response Volume', fontsize=12)
ax1.set_ylabel('Responses per Week'); ax1.set_xlabel('Week Number')
ax1.legend(framealpha=0.15, labelcolor='white', fontsize=9)
ax1.grid(True, alpha=0.25)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.plot(weekly_counts['week'], cumulative, color='#96EFFF', linewidth=2.2)
ax2.fill_between(weekly_counts['week'], cumulative, alpha=0.1, color='#96EFFF')
ax2.set_title('📊  Cumulative Response Count Over Time', fontsize=12)
ax2.set_ylabel('Total Responses'); ax2.set_xlabel('Week Number')
ax2.grid(True, alpha=0.25)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.text(weekly_counts['week'].max()*0.65, cumulative.max()*0.55,
         f'Total: {total:,} responses', color='#96EFFF', fontsize=12, alpha=0.8)

plt.tight_layout()
plt.savefig('chart4_response_trend.png', dpi=150, bbox_inches='tight', facecolor='#0F0F1A')
plt.show()
print("💾 Saved: chart4_response_trend.png")

In [ ]:
# CELL 9 — Demographics: Device + Education + Age
# Three respondent profile charts for survey methodology documentation.
fig = plt.figure(figsize=(15, 6))
fig.patch.set_facecolor('#0F0F1A')
gs2 = gridspec.GridSpec(1, 3, figure=fig, wspace=0.40)

# Device donut
ax_a = fig.add_subplot(gs2[0]); ax_a.set_facecolor('#161625')
dev_counts = df['device'].value_counts()
wedges, texts, autotexts = ax_a.pie(
    dev_counts.values, labels=dev_counts.index, autopct='%1.1f%%',
    colors=['#7C83FD','#96EFFF','#FFD166'], startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0F0F1A', linewidth=1.5),
    textprops={'color':'#C8C8E8','fontsize':10}, pctdistance=0.78)
for at in autotexts: at.set_color('#0F0F1A'); at.set_fontweight('bold')
ax_a.set_title('📱  Responses by Device', color='#E8E8FF')

# Education breakdown
ax_b = fig.add_subplot(gs2[1]); ax_b.set_facecolor('#161625')
edu_counts = df['education'].value_counts().sort_values(ascending=True)
ax_b.barh(edu_counts.index, edu_counts.values,
          color=['#7C83FD','#96EFFF','#FFD166','#EF476F'], alpha=0.88, edgecolor='none')
for bar, val in zip(ax_b.patches, edu_counts.values):
    ax_b.text(val+3, bar.get_y()+bar.get_height()/2,
              f'{val} ({val/total*100:.0f}%)', va='center', color='#C8C8E8', fontsize=9)
ax_b.set_title('🎓  Respondents by Education', color='#E8E8FF')
ax_b.set_xlabel('Count')
ax_b.grid(True, axis='x', alpha=0.25)
ax_b.spines['top'].set_visible(False); ax_b.spines['right'].set_visible(False)

# Age group
ax_c = fig.add_subplot(gs2[2]); ax_c.set_facecolor('#161625')
age_counts = df['age_group'].value_counts().reindex(age_groups)
bars_age = ax_c.bar(age_counts.index, age_counts.values,
                    color=[AGE_COLORS[a] for a in age_groups],
                    alpha=0.88, edgecolor='none', width=0.6)
for b, v in zip(bars_age, age_counts.values):
    ax_c.text(b.get_x()+b.get_width()/2, v+4, f'{v/total*100:.0f}%',
              ha='center', color='#C8C8E8', fontsize=9)
ax_c.set_title('📊  Respondents by Age Group', color='#E8E8FF')
ax_c.set_ylabel('Count')
ax_c.grid(True, axis='y', alpha=0.25)
ax_c.spines['top'].set_visible(False); ax_c.spines['right'].set_visible(False)

fig.suptitle('🔍  Respondent Demographics & Device Analysis', fontsize=14, color='#E8E8FF', y=1.02)
plt.tight_layout()
plt.savefig('chart5_demographics.png', dpi=150, bbox_inches='tight', facecolor='#0F0F1A')
plt.show()
print("💾 Saved: chart5_demographics.png")

In [ ]:
# CELL 10 — Percentage Summary Table
# Clean percentage breakdown for all 5 questions — shareable as text output.
print("\n" + "="*65)
print("  POLL RESULTS — PERCENTAGE BREAKDOWN")
print("="*65)
for i, q in enumerate(q_cols, 1):
    counts = df[q].value_counts().sort_values(ascending=False)
    print(f"\n  {q}")
    print(f"  {'Option':<28} {'Votes':>6} {'%':>7}")
    print(f"  {'-'*44}")
    for opt, cnt in counts.items():
        pct = cnt/total*100
        bar = '█' * int(pct/3)
        print(f"  {opt:<28} {cnt:>6} {pct:>6.1f}%  {bar}")
print("\n" + "="*65)

In [ ]:
# CELL 11 — Auto-Generated Poll Insights
# Human-readable findings generated automatically from the data.
print("💡 AUTO-GENERATED POLL INSIGHTS")
print("="*60)
for q in q_cols:
    counts  = df[q].value_counts()
    winner  = counts.idxmax()
    runnerup = counts.index[1]
    gap     = (counts.iloc[0] - counts.iloc[1]) / total * 100
    print(f"\n  {q}")
    print(f"  → 🥇 Winner   : '{winner}' ({counts.max()/total*100:.1f}%)")
    print(f"  → 🥈 Runner-up: '{runnerup}' ({counts.iloc[1]/total*100:.1f}%)")
    print(f"  → Gap         : {gap:.1f} percentage points")

print(f"\n📍 DEMOGRAPHIC INSIGHTS:")
top_age = df.groupby('age_group')[q_cols[0]].apply(lambda x: x.value_counts().idxmax())
print(f"  • Preferred language by age group:")
for age, lang in top_age.items():
    print(f"    {age}: {lang}")

print(f"\n📍 REGIONAL INSIGHTS:")
top_region = df.groupby('region')[q_cols[1]].apply(lambda x: x.value_counts().idxmax())
print(f"  • Preferred work mode by region:")
for region, mode in top_region.items():
    print(f"    {region}: {mode}")

print(f"\n📍 DEVICE INSIGHTS:")
mob_pct = df['is_mobile'].mean()*100
print(f"  • {mob_pct:.1f}% of respondents used mobile to complete the survey")
print(f"  • This signals the need for mobile-first survey design")

In [ ]:
# CELL 12 — Final Report Summary
# Complete boxed summary — screenshot this for GitHub README.
print()
print("╔══════════════════════════════════════════════════════╗")
print("║       POLL RESULTS VISUALIZER — FINAL REPORT        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  📋 Total Questions   : {len(q_cols):<28}║")
print(f"║  👥 Total Respondents : {total:,}{'':<23}║")
print(f"║  🌍 Regions Covered   : {df['region'].nunique():<28}║")
print(f"║  🎂 Age Groups        : {df['age_group'].nunique():<28}║")
print(f"║  📱 Top Device        : {df['device'].value_counts().idxmax():<28}║")
print(f"║  📅 Survey Period     : Jan 2024 – Oct 2024         ║")
print("╠══════════════════════════════════════════════════════╣")
for i, q in enumerate(q_cols, 1):
    winner = df[q].value_counts().idxmax()
    pct    = df[q].value_counts().max()/total*100
    short_q = f"Q{i} Winner"
    print(f"║  🏆 {short_q:<18}: {winner[:24]:<24} ({pct:.0f}%)  ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  📁 Output Files:                                    ║")
print("║     poll_data.csv                                   ║")
print("║     chart1_poll_dashboard.png                       ║")
print("║     chart2_regional_analysis.png                    ║")
print("║     chart3_demographic_heatmap.png                  ║")
print("║     chart4_response_trend.png                       ║")
print("║     chart5_demographics.png                         ║")
print("╚══════════════════════════════════════════════════════╝")